> # Leak Water


In [0]:
# 1. The "Ingestion & Processing" Layer (PySpark)
# In Databricks, Spark is native. 
# This code simulates receiving "dirty" sensor data from a stream (Kafka) and cleaning it for storage in a Data Lake (Delta Lake/S3).

# --- STEP 1: Spark Data Cleaning (The 'Compute' Phase) ---
from pyspark.sql.functions import col, current_timestamp, avg

# Simulating incoming sensor data (Pressure from Water Network)
data = [("Sensor_A1", 45.2), ("Sensor_A2", 38.5), ("Sensor_A1", 44.8), ("Sensor_B1", 12.0)]
columns = ["sensor_id", "pressure_psi"]

df = spark.createDataFrame(data, columns)
display(df)

In [0]:
# Logic: Identify low pressure (Potential Leak < 40 PSI)
cleaned_df = df.withColumn("ingestion_time", current_timestamp()) \
               .filter(col("pressure_psi") > 0) # Remove noise

# Aggregating to find average pressure by sector
stats_df = cleaned_df.groupBy("sensor_id").agg(avg("pressure_psi").alias("avg_pressure"))

stats_df.show()
# This results would typically be saved to Snowflake or a Delta Table here.

In [0]:
# 2. The "Advanced Intelligence" Layer (Ray)
# While Spark handles the table above, we use Ray to run a parallel simulation or a complex model that doesn't fit into a standard SQL-like table. Note: In Databricks, you can install Ray via %pip install ray.

# --- STEP 2: Ray Distributed Task (The 'Analyze' Phase) ---
!pip install ray
import ray

In [0]:
# Initialize Ray (In Databricks, this connects to the cluster nodes)
if not ray.is_initialized():
    ray.init()

@ray.remote
def predict_leak_risk(sensor_id, avg_pressure):
    """
    Complex logic that might use a pre-trained TensorFlow 
    or Scikit-learn model to predict the probability of a burst.
    """
    import time
    time.sleep(0.5) # Simulating heavy computation
    risk_score = "HIGHT" if avg_pressure < 35 else "LOW"
    return {sensor_id: risk_score}
    

# Execute the function in parallel for all sensors found by Spark
sensor_list = stats_df.collect()
futures = [predict_leak_risk.remote(row['sensor_id'], row['avg_pressure']) for row in sensor_list]

# Get results back
final_risks = ray.get(futures)
print(f"Real-time Risk Assessment: {final_risks}")

In [0]:
# Another alternative
from pyspark.sql.functions import udf, col, avg
from pyspark.sql.types import StringType
import time

# 2. THE CUSTOM LOGIC (The "Brain")
# This is where your AI model or complex math would live
def analyze_risk_logic(pressure):
    """
    Simulates a complex Python model.
    Spark will run this in parallel on all worker nodes.
    """
    if pressure is None: return "UNKNOWN"
    # Imagine calling a scikit-learn .predict() here
    if pressure < 30:
        return "CRITICAL LEAK"
    elif pressure < 40:
        return "WARNING"
    else:
        return "STABLE"

# Register the Python function as a Spark UDF
risk_udf = udf(analyze_risk_logic, StringType())

# 3. THE EXECUTION
# We aggregate the data and THEN apply the intelligence
final_results = df.groupBy("sensor_id") \
                  .agg(avg("pressure_psi").alias("avg_p")) \
                  .withColumn("risk_status", risk_udf(col("avg_p")))

# Display results
print("--- Integrated Spark Risk Intelligence ---")
final_results.show()

# 4. EXPORT TO SQL (For Dashboards)
final_results.createOrReplaceTempView("v_leak_intelligence")

# Bank Example
## Credit Risk

In [0]:
# 1. Ingestion: Upload the credit_risk_dataset.csv to a Unity Catalog Volume.

# * Bronze (Raw): Read the CSV exactly as it is. This is your "Source of Truth."
# * Silver (Cleaned): This is where you handle the specific issues in this Kaggle set (imputing missing interest rates, removing outliers in person age, etc.).
# * Gold (Risk Features): Creating the final "Default Probability" flags for the business.

# 2. Implementation Code (Spark-Native)

# First Step:  
## GO to Create Medallion (schemas created)
$$bronze \rightarrow \rightarrow silver \rightarrow gold$$

In [0]:
# Step A: Bronze (The Raw Table)
# We read from the Volume and save it as a permanent table in the Bronze Schema.
# Path to your uploaded Kaggle file in the Volume
csv_path = "/Volumes/workspace/bronze/landing_zone/credit_risk_dataset.csv"

# Read Raw
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(csv_path)

# Save to Bronze Schema
df_raw.write.mode("overwrite").saveAsTable("workspace.bronze.credit_risk_raw")

In [0]:
# EDA: Credit Risk Raw Data
# 1. Proportion of loans (loan_status: 1 = default, 0 = no default)
# 2. Feature selection (correlation + importance)

from pyspark.sql.functions import col, count, when, round as _round
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load bronze table
df = spark.read.table("workspace.bronze.credit_risk_raw")

# --- Overview ---
print("Shape:", (df.count(), len(df.columns)))
print("\nSchema:")
df.printSchema()

print("\nMissing values per column:")
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# --- 1. Proportion of loan_status (0 vs 1) ---
print("=== Loan Status Distribution ===")
total = df.count()
status_counts = df.groupBy("loan_status") \
    .agg(count("*").alias("count")) \
    .withColumn("proportion", _round(col("count") / total * 100, 2)) \
    .orderBy("loan_status")
status_counts.show()

# Plot the distribution
status_pd = status_counts.toPandas()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(status_pd["loan_status"].astype(str), status_pd["count"], color=["steelblue", "salmon"])
ax.set_xlabel("loan_status (0 = No Default, 1 = Default)")
ax.set_ylabel("Count")
ax.set_title("Loan Status Proportion")
for i, row in status_pd.iterrows():
    ax.text(i, row["count"] + total * 0.01, f'{row["proportion"]}%', ha="center")
plt.tight_layout()
plt.show()

# --- 2. Feature Selection ---

# 2a. Correlation matrix (numeric columns only)
numeric_cols = [f.name for f in df.schema.fields 
                if f.dataType.typeName() in ("integer", "double", "long", "float", "decimal")]

print("Numeric columns:", numeric_cols)

# Convert to pandas for correlation (sample if large)
df_pd = df.select(numeric_cols).sample(False, 0.5, seed=42).toPandas()

corr = df_pd.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Matrix (Numeric Features)")
plt.tight_layout()
plt.show()

# Correlation with target (loan_status) sorted
print("\n=== Correlation with loan_status ===")
if "loan_status" in corr.columns:
    corr_target = corr["loan_status"].drop("loan_status").sort_values(ascending=False)
    print(corr_target)

# 2b. Feature importance via Random Forest
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier

# Identify categorical and numeric feature columns (exclude target)
cat_cols = [f.name for f in df.schema.fields 
            if f.dataType.typeName() in ("string",) and f.name != "loan_status"]
num_cols = [f.name for f in df.schema.fields 
            if f.dataType.typeName() in ("integer", "double", "long", "float", "decimal") 
            and f.name != "loan_status"]

# Index categorical columns
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
assembled_cols = num_cols + [f"{c}_idx" for c in cat_cols]

# Build a simple pipeline
from pyspark.ml import Pipeline
assembler = VectorAssembler(inputCols=assembled_cols, outputCol="features", handleInvalid="skip")
rf = RandomForestClassifier(labelCol="loan_status", featuresCol="features", numTrees=50, seed=42)
pipeline = Pipeline(stages=indexers + [assembler, rf])

# Drop rows with null target for the model
df_model = df.na.drop(subset=["loan_status"])
model = pipeline.fit(df_model)

# Extract feature importances
rf_model = model.stages[-1]
importances = rf_model.featureImportances.toArray()
feature_names = assembled_cols
importance_df = pd.DataFrame({"feature": feature_names, "importance": importances}) \
    .sort_values("importance", ascending=False)

print("\n=== Random Forest Feature Importances ===")
print(importance_df.to_string(index=False))

# Plot feature importances
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=importance_df, x="importance", y="feature", ax=ax, palette="viridis")
ax.set_title("Feature Importances (Random Forest)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

In [0]:
display(df_raw)

In [0]:
%sql
select person_age 
from workspace.bronze.credit_risk_raw;

select * from workspace.bronze.credit_risk_raw
where person_age > 50

In [0]:
# Step B: Silver (The Cleaned Table)
# We apply the "Bank Rules" (handling nulls and filtering outliers) and save it to the Silver Schema.

from pyspark.sql.functions import when, col, mean, split

# Read from Bronze

bronze_df = spark.read.table("workspace.bronze.credit_risk_raw")

# Cleaning Logic: Fill nulls & remove age outliers
avg_rate = bronze_df.select(mean("loan_int_rate")).collect()[0][0]

df_silver = bronze_df.withColumn("loan_int_rate", 
    when(col("loan_int_rate").isNull(), avg_rate).otherwise(col("loan_int_rate"))) \
    .filter(col("person_age") < 100)

# Save to Silver
df_silver.write.mode("overwrite").saveAsTable("workspace.silver.credit_risk_cleaned")

In [0]:
# Layer 3: Gold (The "Knowledge")
# We apply the high-level business logic (Credit Risk segments) and save to gold.
silver_df = spark.read.table("workspace.silver.credit_risk_cleaned")

# Business Logic: Segmenting users by risk
df_gold = silver_df.withColumn("risk_segment", 
    when((col("loan_percent_income") > 0.3) & (col("person_home_ownership") == 'RENT'), "HIGH_RISK")
    .otherwise("STANDARD_RISK")
)

# Save to Gold
df_gold.write.mode("overwrite").saveAsTable("workspace.gold.credit_risk_final")